# Experiment 12: Mini Project — End-to-End Agentic System

## Objective

To build an end-to-end agentic AI system combining RAG, tool use, multi-agent collaboration, and simple deployment.

## System Architecture

User Question
↓
RAG Retriever
↓
Research Agent
↓
Tool Agent
↓
Answer Agent
↓
Final Response
↓
Gradio Web Interface

## Technologies

* Python
* Google Colab
* Sentence Transformers
* FAISS
* SQLite / Python tools
* Gradio


In [1]:
print("Agentic AI Capstone Project")
print("Experiment 12")
print("RAG + Tools + Multi-Agent + Deployment")

Agentic AI Capstone Project
Experiment 12
RAG + Tools + Multi-Agent + Deployment


In [2]:
!pip -q install transformers sentence-transformers faiss-cpu gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 58.9 MB/s eta 0:00:00


In [3]:
documents = [
    "Artificial Intelligence is the field of creating computer systems that can perform tasks that normally require human intelligence.",

    "Machine Learning is a branch of Artificial Intelligence that enables computers to learn patterns from data and make predictions.",

    "Retrieval-Augmented Generation, or RAG, combines document retrieval with language generation. Relevant information is retrieved before generating an answer.",

    "Agentic AI systems use agents that can reason about tasks, use tools, retrieve information, and take actions to achieve a goal.",

    "Multi-agent systems contain multiple specialized agents. Each agent can perform a specific role and collaborate with other agents.",

    "AI tools allow agents to interact with resources such as databases, calculators, search systems, and APIs.",

    "Model deployment makes an AI application available to users through an interface such as a web application or API."
]

print("Knowledge base created successfully!")
print("Number of documents:", len(documents))

Knowledge base created successfully!
Number of documents: 7


In [4]:
from sentence_transformers import SentenceTransformer

# Load the embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert documents into embeddings
document_embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
)

print("Embeddings created successfully!")
print("Embedding shape:", document_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings created successfully!
Embedding shape: (7, 384)


In [5]:
import faiss
import numpy as np

# Get the embedding dimension
embedding_dimension = document_embeddings.shape[1]

# Create the FAISS index
index = faiss.IndexFlatL2(embedding_dimension)

# Add document embeddings
index.add(
    np.array(document_embeddings).astype("float32")
)

print("FAISS vector store created successfully!")
print("Number of vectors:", index.ntotal)

FAISS vector store created successfully!
Number of vectors: 7


In [6]:
def retrieve_documents(question, top_k=2):
    # Convert the question into an embedding
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    # Search FAISS for relevant documents
    distances, indices = index.search(
        question_embedding,
        top_k
    )

    # Retrieve matching documents
    retrieved_docs = [
        documents[i]
        for i in indices[0]
        if i >= 0
    ]

    return retrieved_docs


# Test the retriever
question = "What is RAG?"

retrieved_docs = retrieve_documents(question)

print("User Question:", question)
print("\nRetrieved Documents:")

for doc in retrieved_docs:
    print("-", doc)

User Question: What is RAG?

Retrieved Documents:
- Retrieval-Augmented Generation, or RAG, combines document retrieval with language generation. Relevant information is retrieved before generating an answer.
- Multi-agent systems contain multiple specialized agents. Each agent can perform a specific role and collaborate with other agents.


In [7]:
def research_agent(question):
    retrieved_docs = retrieve_documents(question, top_k=2)

    research = {
        "question": question,
        "documents": retrieved_docs
    }

    return research


# Test the Research Agent
research_result = research_agent("What is RAG?")

print("Research Agent Result:")
print("Question:", research_result["question"])

print("\nRetrieved Information:")
for doc in research_result["documents"]:
    print("-", doc)

Research Agent Result:
Question: What is RAG?

Retrieved Information:
- Retrieval-Augmented Generation, or RAG, combines document retrieval with language generation. Relevant information is retrieved before generating an answer.
- Multi-agent systems contain multiple specialized agents. Each agent can perform a specific role and collaborate with other agents.


In [8]:
def calculator_tool(a, b, operation):
    if operation == "add":
        return a + b

    elif operation == "subtract":
        return a - b

    elif operation == "multiply":
        return a * b

    elif operation == "divide":
        if b == 0:
            return "Cannot divide by zero"
        return a / b

    else:
        return "Unknown operation"


# Test the calculator
result = calculator_tool(10, 5, "multiply")

print("Calculator Tool Result:", result)

Calculator Tool Result: 50


In [9]:
def tool_agent(a, b, operation):
    result = calculator_tool(a, b, operation)

    return {
        "tool": "calculator",
        "input": {
            "a": a,
            "b": b,
            "operation": operation
        },
        "result": result
    }


# Test the Tool Agent
tool_result = tool_agent(20, 4, "divide")

print("Tool Agent Result:")
print(tool_result)

Tool Agent Result:
{'tool': 'calculator', 'input': {'a': 20, 'b': 4, 'operation': 'divide'}, 'result': 5.0}


In [10]:
def answer_agent(question, research_result, tool_result=None):
    answer = f"Question: {question}\n\n"

    answer += "Relevant Information:\n"

    for doc in research_result["documents"]:
        answer += "- " + doc + "\n"

    if tool_result is not None:
        answer += "\nTool Result:\n"
        answer += str(tool_result["result"])

    return answer


# Test the Answer Agent
question = "What is RAG?"

research_result = research_agent(question)

final_response = answer_agent(
    question,
    research_result
)

print("Final Answer:")
print(final_response)

Final Answer:
Question: What is RAG?

Relevant Information:
- Retrieval-Augmented Generation, or RAG, combines document retrieval with language generation. Relevant information is retrieved before generating an answer.
- Multi-agent systems contain multiple specialized agents. Each agent can perform a specific role and collaborate with other agents.



In [11]:
def multi_agent_system(question):
    # Step 1: Research Agent
    research_result = research_agent(question)

    # Step 2: Tool Agent
    tool_result = None

    # Step 3: Answer Agent
    final_response = answer_agent(
        question,
        research_result,
        tool_result
    )

    return final_response


# Test the complete system
question = "What is RAG?"

result = multi_agent_system(question)

print("Final System Response:")
print(result)

Final System Response:
Question: What is RAG?

Relevant Information:
- Retrieval-Augmented Generation, or RAG, combines document retrieval with language generation. Relevant information is retrieved before generating an answer.
- Multi-agent systems contain multiple specialized agents. Each agent can perform a specific role and collaborate with other agents.



In [12]:
test_questions = [
    "What is Artificial Intelligence?",
    "What is Machine Learning?",
    "What is RAG?",
    "What are multi-agent systems?",
    "What is model deployment?"
]

for question in test_questions:
    print("=" * 60)
    print("Question:", question)

    result = multi_agent_system(question)

    print("\nAgent Response:")
    print(result)
    print()

Question: What is Artificial Intelligence?

Agent Response:
Question: What is Artificial Intelligence?

Relevant Information:
- Artificial Intelligence is the field of creating computer systems that can perform tasks that normally require human intelligence.
- Machine Learning is a branch of Artificial Intelligence that enables computers to learn patterns from data and make predictions.


Question: What is Machine Learning?

Agent Response:
Question: What is Machine Learning?

Relevant Information:
- Machine Learning is a branch of Artificial Intelligence that enables computers to learn patterns from data and make predictions.
- Artificial Intelligence is the field of creating computer systems that can perform tasks that normally require human intelligence.


Question: What is RAG?

Agent Response:
Question: What is RAG?

Relevant Information:
- Retrieval-Augmented Generation, or RAG, combines document retrieval with language generation. Relevant information is retrieved before generat

In [13]:
%pip install -q gradio

In [14]:
import gradio as gr

def chatbot(question):
    if not question.strip():
        return "Please enter a question."

    return multi_agent_system(question)


demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(
        label="Ask a Question",
        placeholder="What is RAG?"
    ),
    outputs=gr.Textbox(
        label="Agent Response"
    ),
    title="Agentic AI Capstone Project",
    description="Ask questions using RAG and a multi-agent system."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://361d7e8ee69b24b2e3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Experiment 12: Mini Project — End-to-End Agentic System

## Objective

To develop an end-to-end Agentic AI system integrating Retrieval-Augmented Generation (RAG), tool use, multi-agent collaboration, and simple deployment.

## Technologies Used

* Python
* Google Colab
* Sentence Transformers
* FAISS
* Gradio

## Modules Implemented

1. Knowledge Base Creation
2. Text Embedding Generation
3. FAISS Vector Store
4. RAG Document Retrieval
5. Research Agent
6. Calculator Tool
7. Tool Agent
8. Answer Agent
9. Multi-Agent Workflow
10. Gradio Web Interface

## System Architecture

User Question
↓
RAG Retriever
↓
Research Agent
↓
Tool Agent
↓
Answer Agent
↓
Final Response
↓
Gradio Web Interface

## Result

A prototype Agentic AI system was implemented to retrieve relevant information from a knowledge base, organize responses using specialized agents, and provide a web interface for user interaction.

## Conclusion

This experiment demonstrated how RAG, Python tools, specialized agents, and Gradio can be combined to build a simple end-to-end Agentic AI prototype.
